In [ ]:
!pip install bert-score

## Unified model

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW
from bert_score import score as bert_score
import re

class SelfImprovingModel:
    def __init__(self, model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0", network=None, model_path=None, lr=5e-5, device=None, penalty_factor=1.0):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        self.model_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path if model_path else model_name,
            torch_dtype=self.model_dtype,
            device_map="auto"
        )

        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.optimizer = AdamW(self.model.parameters(), lr=lr)
        self.last_score = 0.0
        self.penalty_factor = penalty_factor
        self.feedback_prompt_condenser = GPTPromptFeedbackGenerator()
        self.network = network

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def generate_prompt(self, question, previous_answer=None, unified_feedback=None, cot=None):
        prompt = ""

        if cot:
            prompt += cot.replace("{replace with the prompt}", question.strip() + "\n\n")
        else:
            prompt += f"Q: {question.strip()}\n"

        if previous_answer:
            if isinstance(previous_answer, tuple):
                previous_answer = previous_answer[1]
            prompt += f"\nPrevious Answer:\n{previous_answer.strip()}\n"

            if self.last_score:
                prompt += f"\nThe previous answer received a score of {self.last_score} on a scale from -1.0 to 1.0."

            if unified_feedback:
                prompt += f"\n\nFeedback:\n{unified_feedback.strip()}"
            else:
                prompt += "\n\n(Feedback not available yet. Try to improve the answer based on clarity, accuracy, and completeness.)"

            prompt += (
                "\n\nYour task is to generate an improved answer using the feedback and prior response as reference."
                "\n- Focus on clarity, accuracy, and logical completeness."
                "\n- Build upon strengths of the previous answer."
                "\n- Ensure the response is coherent and self-contained.\n"
                "\nAnswer:\nA:"
            )
        else:
            # No previous answer – generate from scratch
            prompt += (
                "\nThere is no previous answer available.\n"
                "Please generate a new answer to the question above."
                "\n- Aim for clarity, accuracy, and completeness."
                "\n- Ensure the answer is logically structured and self-contained.\n"
                "\nAnswer:\nA:"
            )

        return prompt



    def generate_answer(self, question, previous_answer=None, previous_score=0.0, previous_feedback=None, cot=None, max_new_tokens=100):
        self.last_score = previous_score

        # Step 1: If previous_answer exists, generate feedback
        if self.network and previous_answer is not None:
            initial_feedback = self.network.generate_combined_feedback(question, previous_answer)
            expert_feedback = initial_feedback.get("expert_feedback")
            amateur_feedback = initial_feedback.get("amateur_feedback")
            unified_feedback = self.feedback_prompt_condenser.generate_unified_feedback(
                expert_feedback, amateur_feedback, self.last_score
            )
        else:
            unified_feedback = previous_feedback or "No feedback provided."

        # Step 2: Construct prompt for revision using unified feedback
        prompt = self.generate_prompt(
            question=question,
            previous_answer=previous_answer,
            unified_feedback=unified_feedback,
            cot=cot
        )

        # Step 3: Generate a revised answer
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        output = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=1.0,
            top_p=1.0,
            do_sample=False
        )

        generated_tokens = output[0][inputs["input_ids"].shape[-1]:]  # Only decode the generated continuation
        revised_answer = self.tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


        # Step 4: Generate feedback for the revised answer
        final_feedback = self.network.generate_combined_feedback(question, revised_answer)
        expert_feedback = final_feedback.get("expert_feedback")
        amateur_feedback = final_feedback.get("amateur_feedback")

        # Step 5: Condense feedback again based on improved score
        improved_score = final_feedback.get("combined_score", 0.0)
        unified_feedback = self.feedback_prompt_condenser.generate_unified_feedback(
            expert_feedback, amateur_feedback, improved_score
        )

        return unified_feedback, revised_answer, improved_score



    def prepare_inputs_and_labels(self, prompt, answer, score, feedback):
        full_text = (
            "You are an expert evaluator. Your task is to score the quality of the following answer based "
            "on how well it responds to the given prompt.\n"
            "Evaluate the answer on a scale from -1 to 1, where:\n"
            "-1 = completely incorrect, irrelevant, or does not address the prompt\n"
            "0 = neutral or partially addresses the prompt\n"
            "1 = excellent, fully correct, detailed, and thoroughly addresses the prompt\n\n"
            "Respond exactly in this format:\n"
            f"Prompt: {prompt}\n"
            f"Answer: {answer}\n"
            f"Score: {score}\n"
            f"Feedback: {feedback}\n"
        )

        tokenized = self.tokenizer(full_text, return_tensors="pt", truncation=True, max_length=4096)
        input_ids = tokenized.input_ids[0].clone()
        labels = input_ids.clone()

        score_token_ids = self.tokenizer("Score:", add_special_tokens=False).input_ids
        score_token_ids = torch.tensor(score_token_ids, device=input_ids.device)

        start_idx = None
        for i in range(len(input_ids) - len(score_token_ids) + 1):
            if torch.equal(input_ids[i:i + len(score_token_ids)], score_token_ids):
                start_idx = i
                break

        if start_idx is None:
            start_idx = 0

        labels[:start_idx] = -100

        return (
            input_ids.unsqueeze(0).to(self.device),
            labels.unsqueeze(0).to(self.device),
            tokenized.attention_mask.to(self.device)
        )

    def generate_feedback(self, prompt, answer, max_new_tokens=150):
        self.model.eval()

        input_text = (
            "You are an expert evaluator. Your task is to score the quality of the following answer based "
            "on how well it responds to the given prompt.\n"
            "Evaluate the answer on a scale from -1 to 1, where:\n"
            "-1 = completely incorrect, irrelevant, or does not address the prompt\n"
            "0 = neutral or partially addresses the prompt\n"
            "1 = excellent, fully correct, detailed, and thoroughly addresses the prompt\n\n"
            "Respond exactly in this format:\n"
            f"Prompt: {prompt}\n"
            f"Answer: {answer}\n"
        )

        inputs = self.tokenizer(input_text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                top_p=0.9,
                temperature=0.7,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        generated_text = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)

        pattern = r"Score:\s*([-+]?\d*\.?\d+)\s*Feedback:\s*(.*)"
        match = re.search(pattern, generated_text, re.DOTALL)

        if match:
            score = match.group(1).strip()
            feedback = match.group(2).strip().split("\n")[0]
            return f"Score: {score}\nFeedback: {feedback}"
        else:
            return generated_text.strip()

    def fine_tuning(self, train_dataset, output_dir, stopped=100, epochs=1, print_every=10, lm_loss_weight=0.2, semantic_loss_weight=0.8):
        self.model.train()

        for epoch in range(epochs):
            total_loss = 0.0

            for i, sample in enumerate(train_dataset):
                if i == stopped:
                    break

                prompt = sample.get("prompt", "")
                answer = sample.get("original_answer", "")
                score = sample.get("score", "")
                feedback = sample.get("feedback", "")

                input_ids, labels, attention_mask = self.prepare_inputs_and_labels(prompt, answer, score, feedback)

                outputs = self.model(
                    input_ids=input_ids,
                    labels=labels,
                    attention_mask=attention_mask
                )
                lm_loss = outputs.loss

                predicted_feedback = self.generate_feedback(prompt, answer)
                P, R, F1 = bert_score([predicted_feedback], [feedback], lang="en", verbose=False)
                semantic_loss = 1.0 - F1[0].item()
                semantic_loss = torch.tensor(semantic_loss, dtype=self.model_dtype, device=self.device)

                total_combined_loss = lm_loss_weight * lm_loss + semantic_loss_weight * semantic_loss

                self.optimizer.zero_grad()
                total_combined_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()

                total_loss += total_combined_loss.item()

                if (i + 1) % print_every == 0:
                    print(f"\n--- Sample {i+1} (Epoch {epoch+1}) ---")
                    print(f"Prompt: {prompt}")
                    print(f"Answer: {answer}")
                    print(f"GT Score: {score}")
                    print(f"GT Feedback:\n{feedback}")
                    print(f"\nGenerated Feedback:\n{predicted_feedback}")
                    print(f"LM Loss: {lm_loss.item():.4f}")
                    print(f"Semantic Loss: {semantic_loss.item():.4f}")
                    print(f"Total Loss: {total_combined_loss.item():.4f}")
                    print("-" * 60)

            avg_loss = total_loss / (i + 1)
            print(f"\nEpoch {epoch+1} completed - Avg Loss: {avg_loss:.4f}\n")

        self.model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)

    def update_model_with_feedback(self, question, previous_answer, previous_feedback, cot=None):
        combined_feedback, improved_answer, improved_score = self.generate_answer(
            question, previous_answer, previous_feedback, cot
        )

        prev_score = self.network.generate_combined_feedback(question, previous_answer)["combined_score"]
        delta = improved_score - prev_score
        reward = delta if delta >= 0 else delta * self.penalty_factor

        min_reward_magnitude = 0.05
        if abs(reward) < min_reward_magnitude and reward != 0:
            reward = reward / abs(reward) * min_reward_magnitude

        is_satisfied, critique = self.feedback_prompt_condenser.check_feedback_satisfaction(
            question, previous_answer, combined_feedback, improved_answer
        )
        reward += 0.2 if is_satisfied else -0.2

        print(f"Previous Score: {prev_score:.4f}, Improved Score: {improved_score:.4f}, Reward: {reward:.4f}")
        print(f"Critique: {critique}\n")

        return improved_answer, combined_feedback, improved_score, reward

    def self_critique(self, prompt, previous_answer, previous_feedback, cot=None, iterations=10, target_score=0.95):
        best_answer = previous_answer
        feedback = previous_feedback
        best_score = self.network.generate_combined_feedback(prompt, previous_answer)["combined_score"]

        previous_answers = set()
        previous_answers.add(best_answer)

        for step in range(iterations):
            if best_score >= target_score:
                print(f"Target score reached ({best_score:.4f}), stopping early.")
                break

            improved_answer, feedback, improved_score, reward = self.update_model_with_feedback(
                prompt, best_answer, feedback, cot
            )

            if improved_answer in previous_answers:
                reward -= 0.1
                print("Penalty: Repeated answer detected.")

            previous_answers.add(improved_answer)

            print(f"Step {step}: Reward {reward:.4f}, Score {improved_score:.4f}")
            print(f"Improved Answer:\n{improved_answer}\n")

            if improved_score > best_score:
                best_answer = improved_answer
                best_score = improved_score

        print(f"Final Answer:\n{best_answer}")
        print(f"Final Score: {best_score:.4f}")
        return best_answer, best_score


In [ ]:
import openai

openai.api_key = "sk-xxxxxxxxx"

class GPTPromptFeedbackGenerator:
    def __init__(self, model_name = "gpt-4o"):
        self.model_name = model_name

    def prompt_condense_feedback(self, expert_feedback, amateur_feedback, last_score):
        prompt = (
            f"The previous answer received a score of {last_score} (scale: -1.0 to 1.0), where:\n"
            "-1.0 = completely incorrect or misleading\n"
            " 0.0 = partially helpful or neutral\n"
            " 1.0 = accurate, complete, and helpful\n\n"
            "You are given two sets of feedback on this answer:\n"
            f"- Expert Feedback (more reliable): {expert_feedback.strip()}\n"
            f"- Amateur Feedback: {amateur_feedback.strip()}\n\n"
            "Your task is to **condense** these feedbacks into a single, unified summary that:\n"
            "- Prioritizes the expert feedback\n"
            "- Includes any valid or helpful insights from the amateur feedback\n"
            "- Avoids redundancy or contradiction\n"
            "- Clearly communicates how the answer could be improved (in clarity, completeness, structure, logic, etc.)\n\n"
            "**Final Output:** A concise, constructive, and actionable feedback summary for the answer.\n\n"
            "Unified Feedback:"
        )
        return prompt


    def generate_unified_feedback(self, expert_feedback = None, amateur_feedback = None, last_score = None):

        if not expert_feedback and not amateur_feedback:
            return "No feedback available yet."

        full_prompt = self.prompt_condense_feedback(expert_feedback, amateur_feedback, last_score)

        try:
            response = openai.chat.completions.create(
                model=self.model_name,
                messages=[
                    {"role": "system", "content": "You are a helpful assistant that condenses expert and amateur feedback into a unified improved answer."},
                    {"role": "user", "content": full_prompt}
                ],
                temperature=0.7,
                max_tokens=512
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            print(f"Error generating feedback: {e}")
            return None

    def check_feedback_satisfaction(self, question, previous_answer, feedback, current_answer):
        review_prompt = (
            f"Question: {question}\n\n"
            f"Previous Answer: {previous_answer}\n\n"
            f"Combined Feedback (from expert and amateur evaluators): {feedback}\n\n"
            f"Current Answer: {current_answer}\n\n"
            "Based on the combined feedback, does the current answer appropriately address the most important points raised?\n"
            "Consider both expert and amateur feedback but prioritize the expert perspective. "
            "Your reply should begin with 'Yes' or 'No'. If 'No', briefly explain what is still missing or unclear."
        )

        try:
            response = openai.chat.completions.create(
                model=self.model_name,
                messages=[
                    {"role": "system", "content": "You are a critical reviewer assessing if feedback was properly incorporated."},
                    {"role": "user", "content": review_prompt}
                ],
                temperature=0.5,
                max_tokens=150,
            )

            output = response.choices[0].message.content.strip()
            print(f"Feedback Satisfaction Check:\n{output}\n")
            return output.lower().startswith("yes"), output

        except Exception as e:
            print(f"OpenAI API error: {e}")
            return False, "API Error"

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW
from bert_score import score as bert_score
import re

class SelfImprovingModel:
    def __init__(self, model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0", model_path=None, network=None, lr=5e-5, device=None, penalty_factor=1.0):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        self.model_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path if model_path else model_name,
            torch_dtype=self.model_dtype,
            device_map="auto"
        )

        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.optimizer = AdamW(self.model.parameters(), lr=lr)
        self.last_score = 0.0
        self.penalty_factor = penalty_factor
        self.feedback_prompt_condenser = GPTPromptGenerator()
        self.network = network

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def generate_prompt(self, question, previous_answer=None, unified_feedback=None, cot=None):
        prompt = ""

        if cot:
            prompt += cot.replace("{replace with the prompt}", question.strip() + "\n\n")
        else:
            prompt += f"Q: {question.strip()}\n"

        if previous_answer:
            if isinstance(previous_answer, tuple):
                previous_answer = previous_answer[1]
            prompt += f"Previous Answer:\n{previous_answer.strip()}\n\n"

        if self.last_score:
            prompt += f"The previous answer received a score of {self.last_score} on a scale from -1.0 to 1.0.\n"

        if unified_feedback:
            prompt += f"\n{unified_feedback}\n"
        else:
            prompt += "\n(Feedback not available yet. Try to improve the answer based on clarity, accuracy, and completeness.)\n"

        prompt += (
            "\nYour task is to revise the previous answer, ideally improving it using any feedback available.\n"
            "- Improve clarity, accuracy, and structure.\n"
            "- Preserve useful parts of the original.\n"
            "- End with a logically complete conclusion.\n\n"
            "**Revised Answer:**\nA:"
        )

        return prompt

    def generate_answer(self, question, previous_answer=None, previous_score=0.0, previous_feedback=None, cot=None, max_new_tokens=100):
        self.last_score = previous_score

        expert_feedback = None
        amateur_feedback = None

        if self.network and previous_answer is not None:
            result = self.network.generate_combined_feedback(question, previous_answer)
            expert_feedback = result.get("expert_feedback", None)
            amateur_feedback = result.get("amateur_feedback", None)

        unified_feedback = self.feedback_prompt_condenser.generate_unified_feedback(
            expert_feedback, amateur_feedback, self.last_score
        )

        prompt = self.generate_prompt(
            question=question,
            previous_answer=previous_answer,
            unified_feedback=unified_feedback,
            cot=cot
        )

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        output = self.model.generate(**inputs, max_new_tokens=max_new_tokens)
        answer = self.tokenizer.decode(output[0], skip_special_tokens=True)
        revised_answer = answer.split("**Revised Answer:**")[-1].strip()

        # Generate score & feedback for the revised answer
        feedback_text = self.generate_feedback(question, revised_answer)
        match = re.search(r"Score:\s*([-+]?\d*\.?\d+)", feedback_text)
        improved_score = float(match.group(1)) if match else 0.0

        return unified_feedback, revised_answer, improved_score


    def prepare_inputs_and_labels(self, prompt, answer, score, feedback):
        full_text = (
            "You are an expert evaluator. Your task is to score the quality of the following answer based "
            "on how well it responds to the given prompt.\n"
            "Evaluate the answer on a scale from -1 to 1, where:\n"
            "-1 = completely incorrect, irrelevant, or does not address the prompt\n"
            "0 = neutral or partially addresses the prompt\n"
            "1 = excellent, fully correct, detailed, and thoroughly addresses the prompt\n\n"
            "Respond exactly in this format:\n"
            f"Prompt: {prompt}\n"
            f"Answer: {answer}\n"
            f"Score: {score}\n"
            f"Feedback: {feedback}\n"
        )

        tokenized = self.tokenizer(full_text, return_tensors="pt", truncation=True, max_length=4096)
        input_ids = tokenized.input_ids[0].clone()
        labels = input_ids.clone()

        score_token_ids = self.tokenizer("Score:", add_special_tokens=False).input_ids
        score_token_ids = torch.tensor(score_token_ids, device=input_ids.device)

        start_idx = None
        for i in range(len(input_ids) - len(score_token_ids) + 1):
            if torch.equal(input_ids[i:i + len(score_token_ids)], score_token_ids):
                start_idx = i
                break

        if start_idx is None:
            start_idx = 0

        labels[:start_idx] = -100

        return (
            input_ids.unsqueeze(0).to(self.device),
            labels.unsqueeze(0).to(self.device),
            tokenized.attention_mask.to(self.device)
        )

    def generate_feedback(self, prompt, answer, max_new_tokens=150):
        self.model.eval()

        input_text = (
            "You are an expert evaluator. Your task is to score the quality of the following answer based "
            "on how well it responds to the given prompt.\n"
            "Evaluate the answer on a scale from -1 to 1, where:\n"
            "-1 = completely incorrect, irrelevant, or does not address the prompt\n"
            "0 = neutral or partially addresses the prompt\n"
            "1 = excellent, fully correct, detailed, and thoroughly addresses the prompt\n\n"
            "Respond exactly in this format:\n"
            f"Prompt: {prompt}\n"
            f"Answer: {answer}\n"
        )

        inputs = self.tokenizer(input_text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                top_p=0.9,
                temperature=0.7,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        generated_text = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)

        pattern = r"Score:\s*([-+]?\d*\.?\d+)\s*Feedback:\s*(.*)"
        match = re.search(pattern, generated_text, re.DOTALL)

        if match:
            score = match.group(1).strip()
            feedback = match.group(2).strip().split("\n")[0]
            return f"Score: {score}\nFeedback: {feedback}"
        else:
            return generated_text.strip()

    def fine_tuning(self, train_dataset, output_dir, stopped=100, epochs=1, print_every=10, lm_loss_weight=0.2, semantic_loss_weight=0.8):
        self.model.train()

        for epoch in range(epochs):
            total_loss = 0.0

            for i, sample in enumerate(train_dataset):
                if i == stopped:
                    break

                prompt = sample.get("prompt", "")
                answer = sample.get("original_answer", "")
                score = sample.get("score", "")
                feedback = sample.get("feedback", "")

                input_ids, labels, attention_mask = self.prepare_inputs_and_labels(prompt, answer, score, feedback)

                outputs = self.model(
                    input_ids=input_ids,
                    labels=labels,
                    attention_mask=attention_mask
                )
                lm_loss = outputs.loss

                predicted_feedback = self.generate_feedback(prompt, answer)
                P, R, F1 = bert_score([predicted_feedback], [feedback], lang="en", verbose=False)
                semantic_loss = 1.0 - F1[0].item()
                semantic_loss = torch.tensor(semantic_loss, dtype=self.model_dtype, device=self.device)

                total_combined_loss = lm_loss_weight * lm_loss + semantic_loss_weight * semantic_loss

                self.optimizer.zero_grad()
                total_combined_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()

                total_loss += total_combined_loss.item()

                if (i + 1) % print_every == 0:
                    print(f"\n--- Sample {i+1} (Epoch {epoch+1}) ---")
                    print(f"Prompt: {prompt}")
                    print(f"Answer: {answer}")
                    print(f"GT Score: {score}")
                    print(f"GT Feedback:\n{feedback}")
                    print(f"\nGenerated Feedback:\n{predicted_feedback}")
                    print(f"LM Loss: {lm_loss.item():.4f}")
                    print(f"Semantic Loss: {semantic_loss.item():.4f}")
                    print(f"Total Loss: {total_combined_loss.item():.4f}")
                    print("-" * 60)

            avg_loss = total_loss / (i + 1)
            print(f"\nEpoch {epoch+1} completed - Avg Loss: {avg_loss:.4f}\n")

        self.model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)

    def update_model_with_feedback(self, question, previous_answer, previous_feedback, cot=None):
        combined_feedback, improved_answer, improved_score = self.generate_answer(
            question, previous_answer, previous_feedback, cot
        )

        prev_score = self.network.generate_combined_feedback(question, previous_answer)["combined_score"]
        delta = improved_score - prev_score
        reward = delta if delta >= 0 else delta * self.penalty_factor

        min_reward_magnitude = 0.05
        if abs(reward) < min_reward_magnitude and reward != 0:
            reward = reward / abs(reward) * min_reward_magnitude

        is_satisfied, critique = self.feedback_prompt_condenser.check_feedback_satisfaction(
            question, previous_answer, combined_feedback, improved_answer
        )
        reward += 0.2 if is_satisfied else -0.2

        print(f"Previous Score: {prev_score:.4f}, Improved Score: {improved_score:.4f}, Reward: {reward:.4f}")
        print(f"Critique: {critique}\n")

        return improved_answer, combined_feedback, improved_score, reward

    def self_critique(self, prompt, previous_answer, previous_feedback, cot=None, iterations=10, target_score=0.95):
        best_answer = previous_answer
        feedback = previous_feedback
        best_score = self.network.generate_combined_feedback(prompt, previous_answer)["combined_score"]

        previous_answers = set()
        previous_answers.add(best_answer)

        for step in range(iterations):
            if best_score >= target_score:
                print(f"Target score reached ({best_score:.4f}), stopping early.")
                break

            improved_answer, feedback, improved_score, reward = self.update_model_with_feedback(
                prompt, best_answer, feedback, cot
            )

            if improved_answer in previous_answers:
                reward -= 0.1
                print("Penalty: Repeated answer detected.")

            previous_answers.add(improved_answer)

            print(f"Step {step}: Reward {reward:.4f}, Score {improved_score:.4f}")
            print(f"Improved Answer:\n{improved_answer}\n")

            if improved_score > best_score:
                best_answer = improved_answer
                best_score = improved_score

        print(f"Final Answer:\n{best_answer}")
        print(f"Final Score: {best_score:.4f}")
        return best_answer, best_score

In [ ]:
import openai
import re

class GPTExpertFeedbackModel(BaseMockFeedbackModel):
    def __init__(self, model_name="gpt-4o"):
        super().__init__(name="expert")
        self.model_name = model_name

    def predict_score_and_feedback(self, prompt_text, answer):
        system_message = {
            "role": "system",
            "content": (
                "You are an expert evaluator. Your task is to score the quality of the following answer based "
                "on how well it responds to the given prompt.\n\n"
                "Use the following detailed scoring scale:\n"
                "1.0: Perfect completion with no issues.\n"
                "0.5 – 0.9: Generally good, minor flaws.\n"
                "0.1 – 0.4: Partially helpful, moderate issues.\n"
                "0.0: Neutral or off-topic, neither helpful nor harmful.\n"
                "-0.1 – -0.4: Mildly harmful or misleading.\n"
                "-0.5 – -0.9: Significantly off-task or misleading.\n"
                "-1.0: Complete failure or harmful content.\n\n"
                "Provide your reasoning and feedback in no more than 50 words, focusing on what was done well and what was missed.\n"
                "Respond exactly in this format:\n"
                "[reason]xxxx (MAX 50 words). Score: [score]\n\n"
            )
        }

        user_message = {
            "role": "user",
            "content": f"Prompt: {prompt_text}\nAnswer: {answer}"
        }

        try:
            response = openai.chat.completions.create(
                model=self.model_name,
                messages=[system_message, user_message],
                temperature=0,
                max_tokens=150,
                n=1
            )
            output = response.choices[0].message.content.strip()
            print(f"[Expert Model Feedback]\n{output}\n")

            # Extract score
            score_match = re.search(r"Score\s*:\s*(-?\d+\.?\d*)", output, re.IGNORECASE)
            score = float(score_match.group(1)) if score_match else 0.0

            # Extract feedback (text before "Score")
            feedback = output.split("Score")[0].strip()

            self.expert_datasets.append({
                "prompt": prompt_text,
                "answer": answer,
                "score": score,
                "feedback": feedback
            })

        except Exception as e:
            score = 0.0
            feedback = f"Error parsing GPT expert feedback: {e}"

        return score, feedback

In [ ]:
import openai
import re

class GPTAmateurFeedbackModel(BaseMockFeedbackModel):
    def __init__(self, model_name="gpt-3.5-turbo"):
        super().__init__(name="amateur")
        self.model_name = model_name

    def predict_score_and_feedback(self, prompt_text, answer):
        system_message = {
            "role": "system",
            "content": (
                "You are an amateur evaluator learning how to assess answers. Your job is to give simple feedback "
                "and assign a rough score based on how helpful or understandable the answer is.\n\n"
                "Use this scoring scale:\n"
                "1.0: Great answer, easy to understand.\n"
                "0.5 – 0.9: Pretty good, some things could be clearer.\n"
                "0.1 – 0.4: Okay, but a bit confusing or missing things.\n"
                "0.0 or below: Not helpful, or doesn't make sense.\n\n"
                "Try your best to explain in everyday language what worked and what didn't.\n"
                "Respond in this format:\n"
                "[short feedback] Score: [score]\n\n"
            )
        }

        user_message = {
            "role": "user",
            "content": f"Prompt: {prompt_text}\nAnswer: {answer}"
        }

        try:
            response = openai.chat.completions.create(
                model=self.model_name,
                messages=[system_message, user_message],
                temperature=0.7,
                max_tokens=150
            )

            output = response.choices[0].message.content.strip()
            print(f"[Amateur Model Feedback]\n{output}\n")

            # Extract score
            score_match = re.search(r"Score\s*:\s*(-?\d+\.?\d*)", output, re.IGNORECASE)
            score = float(score_match.group(1)) if score_match else 0.0

            # Extract feedback
            feedback = output.split("Score")[0].strip()

            self.expert_datasets.append({
                "prompt": prompt_text,
                "answer": answer,
                "score": score,
                "feedback": feedback
            })

        except Exception as e:
            score = 0.0
            feedback = f"Error parsing GPT amateur feedback: {str(e)}"

        return score, feedback

In [ ]:
class MockAmateurExpertFeedbackNetWork:
    def __init__(self):
        self.teacher = GPTExpertFeedbackModel()
        self.student = GPTAmateurFeedbackModel()

    def generate_combined_feedback(self, prompt_text, answer, alpha=0.8):
        # Get individual scores and feedback
        student_score, student_feedback = self.student.predict_score_and_feedback(prompt_text, answer)
        teacher_score, teacher_feedback = self.teacher.predict_score_and_feedback(prompt_text, answer)

        # Weighted average score favoring expert
        combined_score = alpha * teacher_score + (1 - alpha) * student_score

        # Combined text (for revision prompt)
        combined_feedback = (
            f"The expert highlights: {teacher_feedback}\n"
            f"The amateur also adds: {student_feedback}"
        )

        return {
            "combined_score": combined_score,
            "combined_feedback": combined_feedback,
            "expert_feedback": teacher_feedback,
            "amateur_feedback": student_feedback

In [ ]:
if __name__ == "__main__":
    dummy_network = MockAmateurExpertFeedbackNetWork()
    model = SelfImprovingModel(network=dummy_network)

    question = "Why do you want to become a software engineer?"
    previous_answer = "I enjoy solving problems and building things that can help people."
    previous_feedback = "This is a good start, but it's a bit generic. Try adding a personal story or specific goals to make it more compelling."


    print("\n=== Initial Answer ===")
    print(previous_answer)

    print("\n=== Starting Self-Critique ===")
    final_answer, final_score = model.self_critique(
        prompt=question,
        previous_answer=previous_answer,
        previous_feedback=previous_feedback,
        iterations=10,
        target_score=0.9
    )

    print("\n=== Final Answer ===")
    print(final_answer)
    print(f"Final Score: {final_score:.4f}")

## **Main Experimental Loop - Text Generation**

In [ ]:
!pip install datasets
!pip install transformers
!pip install evaluate
!pip install bert-score
!pip install matplotlib
!pip install seaborn
!pip install nltk

In [ ]:
!huggingface-cli login

In [ ]:
from huggingface_hub import login
login(token="hf_xxxxxxxx")  # Replace with your HF token

In [ ]:
import torch
from datasets import load_dataset, Dataset
from transformers import pipeline
from bert_score import score as bert_score
from nltk.translate.bleu_score import sentence_bleu
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import gc
from tqdm import tqdm

from transformers import pipeline
import torch
import gc

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

# Load base model (TinyLLaMA)
base_generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    device_map="auto"
)
clear_gpu()

# Load amateur model (LLaMA 2 7B Chat)
amateur_generator = pipeline(
    "text-generation",
    model="meta-llama/Llama-2-7b-chat-hf",
    torch_dtype=torch.float16,
    device_map="auto"
)


# clear GPU after use
del base_generator
clear_gpu()


results = {
    "HellaSwag": {"base": [], "amateur": []},
    "TruthfulQA": {"base": [], "amateur": []},
    "ARC-Challenge": {"base": [], "amateur": []},
    "CommonSenseQA": {"base": [], "amateur": []}
}

#Metric Utilities
def get_accuracy(preds, labels):
    return sum(p == l for p, l in zip(preds, labels)) / len(preds)

def get_bert_score(preds, refs):
    P, R, F1 = bert_score(preds, refs, lang="en", verbose=False)
    return F1.tolist()

def get_bleu(pred, ref):
    return sentence_bleu([ref.split()], pred.split())

def greedy_choice(generated, choices):
    return max(choices, key=lambda c: bert_score([generated], [c], lang="en")[2][0].item())


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Device set to use cuda:0


evaluation code


In [ ]:
def run_eval(model, dataset, task_name, mode="generation"):
    outputs, refs, accs, bleu_scores, bert_scores = [], [], [], [], []

    grouped = {}
    for ex in dataset:
        key = ex["question"]
        if key not in grouped:
            grouped[key] = []
        grouped[key].append(ex)

    for q, group in tqdm(grouped.items(), desc=f"Evaluating {task_name}"):
        gold = None
        choices = []
        for ex in group:
            if ex.get("label") == True:
                gold = ex["option"]
            choices.append(ex["option"])

        try:
            generated = model(q)[0]["generated_text"].strip()
        except:
            generated = "N/A"

        outputs.append(generated)
        refs.append(gold)

        if mode == "mcq":
            pred_choice = greedy_choice(generated, choices)
            accs.append(pred_choice.strip().lower() == gold.strip().lower())
        else:
            bleu_scores.append(get_bleu(generated, gold))
            bert_scores.append(get_bert_score([generated], [gold])[0])

    metrics = {}
    if accs: metrics["accuracy"] = sum(accs) / len(accs)
    if bleu_scores: metrics["BLEU"] = sum(bleu_scores) / len(bleu_scores)
    if bert_scores: metrics["BERTScore"] = sum(bert_scores) / len(bert_scores)

    return outputs, metrics

dataset loaders

In [ ]:
def run_eval(model, dataset, task_name, mode="generation"):
    outputs, refs, accs, bleu_scores, bert_scores = [], [], [], [], []

    grouped = {}
    for ex in dataset:
        key = ex["question"]
        if key not in grouped:
            grouped[key] = []
        grouped[key].append(ex)

    for q, group in tqdm(grouped.items(), desc=f"Evaluating {task_name}"):
        gold = None
        choices = []
        for ex in group:
            if ex.get("label") == True:
                gold = ex["option"]
            choices.append(ex["option"])

        try:
            generated = model(q)[0]["generated_text"].strip()
        except:
            generated = "N/A"

        outputs.append(generated)
        refs.append(gold)

        if mode == "mcq":
            pred_choice = greedy_choice(generated, choices)
            accs.append(pred_choice.strip().lower() == gold.strip().lower())
        else:
            bleu_scores.append(get_bleu(generated, gold))
            bert_scores.append(get_bert_score([generated], [gold])[0])

    metrics = {}
    if accs: metrics["accuracy"] = sum(accs) / len(accs)
    if bleu_scores: metrics["BLEU"] = sum(bleu_scores) / len(bleu_scores)
    if bert_scores: metrics["BERTScore"] = sum(bert_scores) / len(bert_scores)

    return outputs, metrics



**FULL EVAL**

In [ ]:
def evaluate_all_datasets(student_model, base_model):
    datasets = {
        "HellaSwag": (load_hellaswag(), "mcq"),
        "TruthfulQA": (load_truthfulqa(), "generation"),
        "ARC-Challenge": (load_arc_challenge(), "mcq"),
        "CommonSenseQA": (load_commonsenseqa(), "mcq")
    }

    all_metrics = {"Base": {}, "Amateur": {}}

    for task_name, (data, mode) in datasets.items():
        print(f"\n--- Running {task_name} ---")
        _, base_metrics = run_eval(base_model, data, task_name, mode)
        _, stu_metrics = run_eval(student_model, data, task_name, mode)

        all_metrics["Base"][task_name] = base_metrics
        all_metrics["Amateur"][task_name] = stu_metrics

    return all_metrics


**VISUALIZATION**

In [ ]:
def plot_bar_metrics(all_metrics):
    tasks = list(all_metrics["Base"].keys())
    metric_names = list(all_metrics["Base"][tasks[0]].keys())

    for metric in metric_names:
        base_scores = [all_metrics["Base"][task].get(metric, 0) for task in tasks]
        stu_scores = [all_metrics["Amateur"][task].get(metric, 0) for task in tasks]

        x = np.arange(len(tasks))
        width = 0.35

        plt.figure(figsize=(10, 6))
        plt.bar(x - width/2, base_scores, width, label="Base")
        plt.bar(x + width/2, stu_scores, width, label="Amateur")
        plt.xticks(x, tasks)
        plt.ylabel(metric)
        plt.title(f"{metric} Comparison: Base vs Amateur")
        plt.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
metrics = evaluate_all_datasets(amateur_generator, base_generator)
plot_bar_metrics(metrics)

## **TASK MITIGATION**

**IMPORTS**

In [ ]:
import random
import matplotlib.pyplot as plt
from transformers import pipeline
from datasets import load_dataset
from detoxify import Detoxify

**LOAD DATASETS**

In [ ]:
import json
from datasets import Dataset

# Load manually as a list of dicts
with open("prompts.json", "r") as f:
    data = [json.loads(line) for line in f]

# Convert to Hugging Face dataset
tox_dataset = Dataset.from_list(data)

In [ ]:
tox_dataset.features
tox_dataset[0]

**LOAD BASE + AMATEUR MODELS**

In [ ]:
from transformers import pipeline
import torch
import gc

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

# load base model (tinyllama)
base_generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    device_map="auto"
)

# clear GPU after use
del base_generator
clear_gpu()

# load amateur model (llama 2 7b chat)
amateur_generator = pipeline(
    "text-generation",
    model="meta-llama/Llama-2-7b-chat-hf",
    torch_dtype=torch.float16,
    device_map="auto"
)

**TOXICITY EVALUATION**

In [ ]:
import json
from datasets import Dataset

with open("prompts.json", "r") as f:
    data = [json.loads(line) for line in f]

tox_dataset = Dataset.from_list(data)

In [ ]:
!pip install -q transformers datasets detoxify accelerate

from transformers import pipeline
from datasets import load_dataset
from detoxify import Detoxify
import torch

device = 0 if torch.cuda.is_available() and torch.cuda.memory_allocated(0) < 8e9 else -1

tox_prompts = [example["prompt"]["text"] for example in list(tox_dataset.select(range(100)))]

print(f"✅ Loaded {len(tox_prompts)} prompts.")
print(tox_prompts[:3])  # preview

In [ ]:
!pip install tqdm

In [ ]:
from tqdm import tqdm

def evaluate_toxicity(model_pipeline, prompts, model_name="Model"):
    outputs = []

    for prompt in tqdm(prompts, desc=f"{model_name} Evaluating"):
        try:
            with torch.no_grad():
                response = model_pipeline(prompt, max_new_tokens=30, do_sample=False)[0]["generated_text"]
            tox_score = Detoxify("original").predict(response)["toxicity"]
            outputs.append((prompt, response, tox_score))
        except Exception as e:
            print(f"❌ Error on prompt: {prompt[:30]}... – {e}")
            continue
    if not outputs:
        print(f"⚠️ {model_name}: No outputs generated — check prompts.")
        return outputs, {"avg_toxicity": None}

    avg_score = sum(score for _, _, score in outputs) / len(outputs)
    print(f"{model_name} Average Toxicity: {avg_score:.4f}")
    return outputs, {"avg_toxicity": avg_score}

tox_outputs_base, tox_base_metrics = evaluate_toxicity(base_generator, tox_prompts, model_name="TinyLlama")
tox_outputs_amateur, tox_amateur_metrics = evaluate_toxicity(amateur_generator, tox_prompts, model_name="LLaMA 2")

**VISUALIZATION**

In [ ]:
def plot_toxicity_metrics(base_metrics, amateur_metrics, base_name="TinyLlama", amateur_name="LLaMA 2"):
    labels = [base_name, amateur_name]
    scores = [base_metrics["avg_toxicity"], amateur_metrics["avg_toxicity"]]

    plt.figure(figsize=(6, 4))
    plt.bar(labels, scores, color=["blue", "green"])
    plt.ylabel("Average Toxicity Score")
    plt.title("Toxicity Mitigation Performance")
    plt.ylim(0, 1)
    plt.grid(axis='y')
    plt.show()

In [ ]:
plot_toxicity_metrics(tox_base_metrics, tox_amateur_metrics)

## **MATHEMATICAL REASONING**

**IMPORTS**
*make sure to login to huggingface

In [ ]:
!pip install datasets evaluate --quiet

from datasets import load_dataset
from evaluate import load as load_metric
from tqdm import tqdm
import numpy as np

In [ ]:
!huggingface-cli login

In [ ]:
from huggingface_hub import login
login(token="hf_xxxxxxxx")  # Replace with your HF token

**LOAD MODELS**

In [ ]:
from transformers import pipeline
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

base_model = pipeline(
    "text-generation",
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

amateur_model = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    device_map="auto"
)

**DATASET LOADERS**

In [ ]:
def load_gsm8k_subset():
    ds = load_dataset("gsm8k", "main", split="test[:100]")  # Smaller sample for speed
    return [{"question": ex["question"], "answer": ex["answer"]} for ex in ds]

def load_mmlu_math_subset():
    ds = load_dataset("cais/mmlu", "abstract_algebra", split="test[:100]")
    return [{"question": ex["input"], "answer": ex["target"]} for ex in ds]

**METRICS**

In [ ]:
accuracy_metric = load_metric("accuracy")

def get_accuracy(preds, labels):
    correct = [1 if label.strip() in pred.strip() else 0 for pred, label in zip(preds, labels)]
    return sum(correct) / len(correct)

**EVALUATION FUNCTION**

In [ ]:
def run_math_eval(model, dataset, task_name):
    preds, labels = [], []

    for i, ex in enumerate(tqdm(dataset, desc=f"🧠 Evaluating {task_name}")):
        print(f"🧠 {task_name} | Prompt {i+1}/{len(dataset)}")

        prompt = ex["question"]
        gold = ex["answer"]

        try:
            response = model(prompt, max_new_tokens=50, do_sample=False)[0]["generated_text"]
        except Exception as e:
            print(f"⚠️ Error on prompt: {prompt[:30]}... — {e}")
            response = "N/A"

        preds.append(response.strip())
        labels.append(gold.strip())

    return get_accuracy(preds, labels)

**FULL EVALUATION**

In [ ]:
def evaluate_math_all(student_model, base_model):
    datasets = {
        "GSM8K": load_gsm8k_subset(),
        "MMLU-Math": load_mmlu_math_subset()
    }

    metrics = {
        "Base": {},
        "Amateur": {}
    }

    for task_name, data in datasets.items():
        base_acc = run_math_eval(base_model, data, task_name)
        stu_acc = run_math_eval(student_model, data, task_name)
        metrics["Base"][task_name] = base_acc
        metrics["Amateur"][task_name] = stu_acc

    return metrics

**VISUALIZATION**

In [ ]:
def plot_math_results(all_metrics):
    import matplotlib.pyplot as plt
    import seaborn as sns

    tasks = list(all_metrics["Base"].keys())
    base_scores = [all_metrics["Base"][task] for task in tasks]
    stu_scores = [all_metrics["Amateur"][task] for task in tasks]

    x = np.arange(len(tasks))
    width = 0.35

    plt.figure(figsize=(10, 6))
    plt.bar(x - width/2, base_scores, width, label='Base')
    plt.bar(x + width/2, stu_scores, width, label='Amateur')
    plt.xticks(x, tasks)
    plt.ylabel("Accuracy")
    plt.title("📊 Mathematical Reasoning: Base vs Amateur")
    plt.legend()
    plt.show()

In [ ]:
all_math_metrics = evaluate_math_all(amateur_model, base_model)
plot_math_results(all_math_metrics)

# **BENCHMARKS**

## **HELM BENCHMARK**


**Purpose**:
Evaluates language models across diverse tasks like text generation, mathematical reasoning, and toxicity mitigation. It ensures fair comparisons across models by using consistent datasets, metrics, and configurations.

**How it Works:**

* You clone and install the HELM repository locally.

* HELM provides a registry of benchmark tasks (e.g., `truthful_qa`, `gsm8k`, `real_toxicity_prompts`).

* For each task, you define a `RunSpec` that configures which model to use, with what parameters (e.g., temperature, number of outputs).

* You then run all benchmark tasks using a centralized runner, which loads the appropriate dataset, applies the model, and computes metrics.

**Key Components:**

* `RunSpec`: Defines a single benchmark run including the task name, model adapter, and generation parameters.

* `AdapterSpec`: Specifies the backend model (e.g., HuggingFace, OpenAI) and its configuration.

* `BenchmarkRunner`: Executes the benchmarks and collects results.

* `CacheConfig`: Caches intermediate outputs to reduce redundant computations.

**Returns:**

`results`: A list of results per task, each containing metrics (accuracy, F1, etc.) and generation logs.

`metrics`: Automatically calculated scores depending on the task (e.g., ROUGE for summarization, accuracy for QA, etc.)

In [ ]:
!git clone https://github.com/stanford-crfm/helm.git


fatal: destination path 'helm' already exists and is not an empty directory.


In [ ]:
%cd helm


/content/helm


In [ ]:
!pip install -e .
!pip install jsonlines fire


Obtaining file:///content/helm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for crfm-helm (pyproject.toml) ... done
  Created wheel for crfm-helm: filename=crfm_helm-0.5.6-0.editable-py3-none-any.whl size=12340 sha256=f9a7234660239a12ed0e7c279b415659fb28cc04a6a42877cc08613af7c2287a
  Stored in directory: /tmp/pip-ephem-wheel-cache-ud24bs3_/wheels/0d/72/8f/e1e76ad60be71ded440239c743a70285924886e3fb11a442ab
Successfully built crfm-helm
  Attempting uninstall: crfm-helm
    Found existing installation: crfm-helm 0.5.6
    Uninstalling crfm-helm-0.5.6:
      Successfully uninstalled crfm-helm-0.5.6


**DATASET LOADERS**

In [ ]:
def load_gsm8k_subset():
    ds = load_dataset("gsm8k", "main", split="test[:100]")  # Smaller sample for speed
    return [{"question": ex["question"], "answer": ex["answer"]} for ex in ds]

def load_mmlu_math_subset():
    ds = load_dataset("cais/mmlu", "abstract_algebra", split="test[:100]")
    return [{"question": ex["input"], "answer": ex["target"]} for ex in ds]

**METRICS**

In [ ]:
accuracy_metric = load_metric("accuracy")

def get_accuracy(preds, labels):
    correct = [1 if label.strip() in pred.strip() else 0 for pred, label in zip(preds, labels)]
    return sum(correct) / len(correct)

**EVALUATION FUNCTION**

In [ ]:
def run_math_eval(model, dataset, task_name):
    preds, labels = [], []

    for i, ex in enumerate(tqdm(dataset, desc=f"🧠 Evaluating {task_name}")):
        print(f"🧠 {task_name} | Prompt {i+1}/{len(dataset)}")

        prompt = ex["question"]
        gold = ex["answer"]

        try:
            response = model(prompt, max_new_tokens=50, do_sample=False)[0]["generated_text"]
        except Exception as e:
            print(f"⚠️ Error on prompt: {prompt[:30]}... — {e}")
            response = "N/A"

        preds.append(response.strip())
        labels.append(gold.strip())

    return get_accuracy(preds, labels)

**FULL EVALUATION**

In [ ]:
def evaluate_math_all(student_model, base_model):
    datasets = {
        "GSM8K": load_gsm8k_subset(),
        "MMLU-Math": load_mmlu_math_subset()
    }

    metrics = {
        "Base": {},
        "Amateur": {}
    }

    for task_name, data in datasets.items():
        base_acc = run_math_eval(base_model, data, task_name)
        stu_acc = run_math_eval(student_model, data, task_name)
        metrics["Base"][task_name] = base_acc
        metrics["Amateur"][task_name] = stu_acc

    return metrics

**VISUALIZATION**

In [ ]:
def plot_math_results(all_metrics):
    import matplotlib.pyplot as plt
    import seaborn as sns

    tasks = list(all_metrics["Base"].keys())
    base_scores = [all_metrics["Base"][task] for task in tasks]
    stu_scores = [all_metrics["Amateur"][task] for task in tasks]

    x = np.arange(len(tasks))
    width = 0.35

    plt.figure(figsize=(10, 6))
    plt.bar(x - width/2, base_scores, width, label='Base')
    plt.bar(x + width/2, stu_scores, width, label='Amateur')
    plt.xticks(x, tasks)
    plt.ylabel("Accuracy")
    plt.title("📊 Mathematical Reasoning: Base vs Amateur")
    plt.legend()
    plt.show()

In [ ]:
all_math_metrics = evaluate_math_all(amateur_model, base_model)
plot_math_results(all_math_metrics)

## **INFERENCE COST**

* **Purpose**:
Calculates the average time it takes for a language model to generate outputs, helping compare the computational cost (in seconds per prompt) between smaller "amateur" models and larger "expert" models.

* **How it works**:
For each prompt in a given list, the function tokenizes the input using `AutoTokenizer`, feeds it into the model with `generate(...)`, and measures the time taken. The average time across all prompts is returned as the final metric.

* **Why it matters**:
Inference cost is a crucial benchmark when evaluating feedback pipelines or distillation frameworks. If a student model is significantly faster than an expert model while producing similar outputs, it can save compute and API cost in downstream applications.

* **Inputs**:

  * `model`: A preloaded Hugging Face model (e.g., LLaMA-3 8B or TinyLlama)

  * `tokenizer`: The corresponding Hugging Face tokenizer

  * `prompts`: A list of text prompts (e.g., questions or tasks)

  * `max_new_tokens`: Maximum number of tokens to generate (default is 100)

* **Returns**:

The average inference time (in seconds) per prompt for the model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

# Load expert model and tokenizer
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

# Set the model to evaluation mode
model.eval()

# Sample prompts
sample_prompts = [
    "Explain the concept of entropy in simple terms.",
    "Solve: What is the derivative of sin(x) * cos(x)?",
    "Describe how a black hole forms."
]

# Function to measure inference time
def measure_inference_time(model, tokenizer, prompts, max_new_tokens=100):
    total_time = 0
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        start_time = time.time()
        _ = model.generate(**inputs, max_new_tokens=max_new_tokens)
        total_time += time.time() - start_time
    return total_time / len(prompts)

# Example: measuring inference time
expert_time = measure_inference_time(model, tokenizer, sample_prompts)

print(f"🧠 Average inference time (Expert LLaMA 3): {expert_time:.2f} sec/prompt")


## **EDIT DISTANCE**

* **Purpose:**
Measures how much the revised output differs from the original model output by calculating the minimum number of edits (insertions, deletions, or substitutions) required to transform one string into another.

* **How it works**:
This benchmark calculates both the raw edit distance and the normalized edit distance between the original and revised outputs. The normalized score divides the edit distance by the length of the longer string to give a relative comparison.

* **Why it matters**:
Edit Distance helps quantify the degree of change in output after feedback. A lower score generally indicates higher similarity, which is helpful when assessing whether feedback improves fluency or structure without completely altering meaning.

* **Inputs**:

  * `original_outputs`: List of model outputs before feedback

  * `revised_outputs`: Corresponding outputs after feedback

  * Make sure to replace this with your dataset of generations before/after fine-tuning

* **Returns**:

  * `edit_distance`: The average number of edits needed

  * `normalized_edit_distance`: The average proportion of edits, normalized by sequence length



In [ ]:
!pip install python-Levenshtein

In [ ]:
import Levenshtein
import pandas as pd

# Replace with your actual outputs from the model
original_outputs = [
    "the cat sat on the mat.",
    "ai is transforming the world."
]

revised_outputs = [
    "a cat is sitting on the mat.",
    "artificial intelligence is changing the world."
]

# Create a DataFrame
df = pd.DataFrame({
    "original_output": original_outputs,
    "revised_output": revised_outputs
})

# Calculate edit distance
df["edit_distance"] = df.apply(
    lambda row: Levenshtein.distance(row["original_output"], row["revised_output"]),
    axis=1
)

# Calculate normalized edit distance
df["normalized_edit_distance"] = df.apply(
    lambda row: Levenshtein.distance(row["original_output"], row["revised_output"]) /
    max(len(row["original_output"]), len(row["revised_output"])),
    axis=1
)

# Display average metrics
print("Average Edit Distance:", df["edit_distance"].mean())
print("Average Normalized Edit Distance:", df["normalized_edit_distance"].mean())

## **ROGUE**

* ** Purpose:**
Evaluates how similar the revised model outputs are to the original ones by comparing overlapping units like n-grams, word sequences, and word pairs. ROUGE is commonly used in summarization and generative tasks to assess content preservation.

* **How it works:**
This benchmark calculates ROUGE-1, ROUGE-2, and ROUGE-L F1 scores:

  * `ROUGE-1`: Measures unigram (word-level) overlap

  * `ROUGE-2`: Measures bigram (2-word sequence) overlap

  * `ROUGE-L`: Measures longest common subsequence (LCS) between original and revised text

* **Inputs:**

  * `original_outputs`: Outputs before feedback

  * `revised_outputs`: Corresponding outputs after applying feedback

* **Returns:**

  * `rouge1_f1`, `rouge2_f1`, `rougel_f1`: Lists of F1 scores for each metric

  * Mean scores for each type give an overall performance view

* **Note:**
  * Setting `use_stemmer=True` improves robustness by ignoring minor grammatical differences (like tense or plural forms).

In [ ]:
!pip install rouge-score

In [ ]:
from rouge_score import rouge_scorer
import pandas as pd

In [ ]:
original_outputs = [
    "The cat sat on the mat.",
    "AI is transforming the world.",
    # more outputs
]

revised_outputs = [
    "A cat was sitting on the mat.",
    "Artificial intelligence is changing the world.",
    # more outputs
]

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for original, revised in zip(original_outputs, revised_outputs):
    scores = scorer.score(original, revised)
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

# Create a DataFrame
df = pd.DataFrame({
    "original_output": original_outputs,
    "revised_output": revised_outputs,
    "rouge1_f1": rouge1_scores,
    "rouge2_f1": rouge2_scores,
    "rougeL_f1": rougeL_scores
})

In [ ]:
print("Average ROUGE-1 F1 Score:", df["rouge1_f1"].mean())
print("Average ROUGE-2 F1 Score:", df["rouge2_f1"].mean())
print("Average ROUGE-L F1 Score:", df["rougeL_f1"].mean())

# METEOR



*   Purpose:
Measures how closely student-generated feedback matches expert feedback using the METEOR score, helping track whether student feedback improves over iterative refinement. METEOR is especially useful in your feedback distillation framework to measure semantic progress without requiring perfect token match.
*   How it works:
For each version of student feedback:

* The expert feedback and student feedback are both split into words (simple whitespace tokenization).

* single_meteor_score() compares them using a combination of:

 * Exact word matches

 * Stemming (e.g., “produce” ≈ “producing”)

 * Synonym matching (e.g., “make” ≈ “create”)

 * Word order penalty (for scrambled or poorly ordered text)

It outputs a score between 0 and 1 — higher means closer semantic and lexical similarity.

Scores across multiple student attempts (e.g., v1 → v4) are averaged to track improvement.
Inputs:

expert_feedback: The gold-standard expert feedback string.

student_feedbacks: A list of feedback strings generated by the student model over multiple refinement iterations.

* Cons / Limitations:
 * Can give high scores for shallow matches (e.g., same nouns, but missing logic)

 * Doesn’t account for factual accuracy or logical structure



In [ ]:
!pip install nltk

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.translate.meteor_score import single_meteor_score
import pandas as pd


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
meteor_scores = []

for original, revised in zip(original_outputs, revised_outputs):
    score = single_meteor_score(original.split(), revised.split())
    meteor_scores.append(score)


In [ ]:
df = pd.DataFrame({
    "original_output": original_outputs,
    "revised_output": revised_outputs,
    "meteor_score": meteor_scores
})

print(df)
print("\nAverage METEOR Score:", round(df["meteor_score"].mean(), 4))


                                     original_output  \
0  Gravity pulls objects toward the center of the...   
1  Photosynthesis happens when plants absorb wate...   

                                      revised_output  meteor_score  
0  Gravity causes objects to fall toward Earth's ...      0.265957  
1  During photosynthesis, plants convert sunlight...      0.159574  

Average METEOR Score: 0.2128


In [ ]:
from nltk.translate.meteor_score import single_meteor_score
import pandas as pd

# Setup
prompt = "Why do plants need sunlight?"
student_answer = "Because it helps them grow."

expert_feedback = "Explain that plants require sunlight to perform photosynthesis, which produces the food they need to survive."

# Simulated student feedback over 4 iterations
feedback_v1 = "Mention that sunlight helps plants grow."
feedback_v2 = "Explain that plants use sunlight to make food through photosynthesis."
feedback_v3 = "Say that plants need sunlight for photosynthesis, which makes food."
feedback_v4 = "State that plants require sunlight to carry out photosynthesis, which helps them produce the food necessary for survival."



student_feedbacks = [feedback_v1, feedback_v2, feedback_v3, feedback_v4]
meteor_scores = []

# Evaluate each iteration vs expert
for fb in student_feedbacks:
    score = single_meteor_score(expert_feedback.split(), fb.split())
    meteor_scores.append(score)

# Output as DataFrame
df = pd.DataFrame({
    "iteration": ["v1", "v2", "v3", "v4"],
    "student_feedback": student_feedbacks,
    "meteor_score_vs_expert": meteor_scores
})

print(df)
print("\nAverage METEOR Score:", round(df["meteor_score_vs_expert"].mean(), 4))



  iteration                                   student_feedback  \
0        v1           Mention that sunlight helps plants grow.   
1        v2  Explain that plants use sunlight to make food ...   
2        v3  Say that plants need sunlight for photosynthes...   
3        v4  State that plants require sunlight to carry ou...   

   meteor_score_vs_expert  
0                0.100000  
1                0.331890  
2                0.331890  
3                0.597531  

Average METEOR Score: 0.3403


# COMET



In [ ]:
!pip install unbabel-comet


In [ ]:
from comet import download_model, load_from_checkpoint

# Step 1: Download and load COMET model
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

# Step 2: Define expert feedback and student feedbacks
expert_feedback = "Explain that plants need sunlight to perform photosynthesis, which is how they make food."

student_feedbacks = [
    "You should talk more about the sun",  # v1 — very vague
    "Say something about how the sun helps",  # v2 — slightly more specific
    "Mention that sunlight is used to make energy in plants",  # v3 — semantically closer
    "Explain that plants use sunlight for photosynthesis to create food"  # v4 — nearly matches expert
]



# Step 3: Evaluate each version using COMET
# Step 3: Evaluate each version using COMET
data = [{"src": "", "mt": s, "ref": expert_feedback} for s in student_feedbacks]

scores = comet_model.predict(data, batch_size=4, gpus=1 if comet_model.device.type == "cuda" else 0)





Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.2. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.11/dist-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_light

In [ ]:
print(scores)
print(type(scores[0]))


Prediction([('scores', [0.45552876591682434, 0.4753819704055786, 0.6633621454238892, 0.8685604929924011]), ('system_score', 0.6157083436846733)])
<class 'list'>


In [ ]:
# Step 4: Print results
# Extract the actual score list from the Prediction object
score_list = dict(scores)["scores"]

# Loop and print each version
for i, (student, score) in enumerate(zip(student_feedbacks, score_list)):
    print(f"\nVersion v{i+1}")
    print(f"Expert Feedback:  {expert_feedback}")
    print(f"Student Feedback: {student}")
    print(f"COMET Score:      {round(score, 4)}")




Version v1
Expert Feedback:  Explain that plants need sunlight to perform photosynthesis, which is how they make food.
Student Feedback: You should talk more about the sun
COMET Score:      0.4555

Version v2
Expert Feedback:  Explain that plants need sunlight to perform photosynthesis, which is how they make food.
Student Feedback: Say something about how the sun helps
COMET Score:      0.4754

Version v3
Expert Feedback:  Explain that plants need sunlight to perform photosynthesis, which is how they make food.
Student Feedback: Mention that sunlight is used to make energy in plants
COMET Score:      0.6634

Version v4
Expert Feedback:  Explain that plants need sunlight to perform photosynthesis, which is how they make food.
Student Feedback: Explain that plants use sunlight for photosynthesis to create food
COMET Score:      0.8686
